In [1]:
# Parameters
base_path = "C:\\TCC_USP"
run_id = "20251126_165935"


In [2]:
# 1. Setup e bibliotecas
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

from src.io import paths
from src.validation import merges
from src.utils.logger import log_result

DATA_PATHS = paths.get_data_paths()
PROC_PATH = str(DATA_PATHS["data_processed"])

paths_map = {
    "labels": os.path.join(PROC_PATH, "labels_y_daily.csv"),
    "oof": os.path.join(PROC_PATH, "16_oof_predictions.csv"),
    "results16": os.path.join(PROC_PATH, "results_16_models_tfidf.json"),
}

for key, path_str in paths_map.items():
    if not os.path.exists(path_str):
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {path_str}")

print(json.dumps(paths_map, indent=2, ensure_ascii=False))


{
  "labels": "C:\\TCC_USP\\data_processed\\labels_y_daily.csv",
  "oof": "C:\\TCC_USP\\data_processed\\16_oof_predictions.csv",
  "results16": "C:\\TCC_USP\\data_processed\\results_16_models_tfidf.json"
}


In [3]:
# 2. Carregar dados e preparar dataframe base
labels = pd.read_csv(paths_map["labels"], parse_dates=["day"])
oof = pd.read_csv(paths_map["oof"], parse_dates=["day"])

if oof.empty:
    raise ValueError("Arquivo 16_oof_predictions.csv está vazio. Rode o notebook 16 primeiro.")

df = oof.merge(
    labels.drop(columns=["y"], errors="ignore"),
    on="day",
    how="left",
    suffixes=("", "_label"),
)

if "ret_next" not in df.columns:
    raise ValueError("Dados não possuem coluna ret_next para validação.")

df["sentiment"] = df["proba"] * 2 - 1

try:
    df["signal_bucket"] = pd.qcut(
        df["sentiment"],
        q=[0, 0.33, 0.66, 1],
        labels=["negativo", "neutro", "positivo"],
        duplicates="drop",
    )
except ValueError:
    df["signal_bucket"] = pd.cut(
        df["sentiment"], bins=3, labels=["negativo", "neutro", "positivo"], include_lowest=True
    )

df.sort_values(["model", "day"], inplace=True)
merges.check_intersection(oof, labels, col_left="day", col_right="day", min_days=90)

display(df.head())
print(df["model"].value_counts())


[day] 2019-08-05 → 2025-11-17 | 1552 dias únicos | 3104 registros | missing=0
[day] 2018-01-02 → 2025-11-19 | 2771 dias únicos | 2771 registros | missing=0
Dias em comum: 1552


,model,fold,row_id,day,y,ret_next,close,proba,row_id_label,ret_next_label,close_label,sentiment,signal_bucket
0,logreg_l2,0,390,2019-08-05,1,0.020640,100098.0,0.496225,570,0.020640,100098.0,-0.007551,negativo
1,logreg_l2,0,391,2019-08-06,1,0.006049,102164.0,0.487361,571,0.006049,102164.0,-0.025278,negativo
2,logreg_l2,0,392,2019-08-07,1,0.012969,102782.0,0.510332,572,0.012969,102782.0,0.020665,neutro
3,logreg_l2,0,393,2019-08-08,0,-0.001143,104115.0,0.464429,573,-0.001143,104115.0,-0.071143,negativo
4,logreg_l2,0,394,2019-08-09,0,-0.020010,103996.0,0.478652,574,-0.020010,103996.0,-0.042697,negativo


model
logreg_l2    1552
rf_200       1552
Name: count, dtype: int64


In [4]:
# 3. Correlações Pearson/Spearman por modelo
corr_rows = []
for model_name, sub in df.groupby("model"):
    sub = sub.dropna(subset=["sentiment", "ret_next"])
    if len(sub) < 3:
        continue
    try:
        pear = stats.pearsonr(sub["sentiment"], sub["ret_next"])
    except Exception:
        pear = (np.nan, np.nan)
    try:
        spear = stats.spearmanr(sub["sentiment"], sub["ret_next"])
    except Exception:
        spear = (np.nan, np.nan)
    corr_rows.append({
        "model": model_name,
        "n": len(sub),
        "pearson_r": pear[0],
        "pearson_p": pear[1],
        "spearman_r": spear[0],
        "spearman_p": spear[1],
    })

corr_df = pd.DataFrame(corr_rows)
display(corr_df)

,model,n,pearson_r,pearson_p,spearman_r,spearman_p
0,logreg_l2,1552,0.012212,0.630699,0.018210,0.473457
1,rf_200,1552,-0.006820,0.788356,-0.005454,0.830001


In [5]:
# 4. ANOVA com buckets de sentimento
anova_rows = []
for model_name, sub in df.groupby("model"):
    valid = sub.dropna(subset=["signal_bucket", "ret_next"])
    groups = [grp["ret_next"].values for _, grp in valid.groupby("signal_bucket") if len(grp) > 0]
    if len(groups) < 2:
        continue
    try:
        f_stat, p_val = stats.f_oneway(*groups)
    except Exception:
        f_stat, p_val = np.nan, np.nan
    means = valid.groupby("signal_bucket")["ret_next"].mean().to_dict()
    anova_rows.append({
        "model": model_name,
        "f_stat": f_stat,
        "p_value": p_val,
        "mean_neg": means.get("negativo", np.nan),
        "mean_neu": means.get("neutro", np.nan),
        "mean_pos": means.get("positivo", np.nan),
        "n_obs": len(valid),
    })

anova_df = pd.DataFrame(anova_rows)
display(anova_df)

if "corr_df" in globals():
    anova_lookup = anova_df.set_index("model").to_dict("index") if not anova_df.empty else {}
    for _, row in corr_df.iterrows():
        model = row.get("model")
        metrics = {"pearson_r": float(row.get("pearson_r", 0) or 0), "spearman_r": float(row.get("spearman_r", 0) or 0)}
        extra = {
            "dataset": "sentiment_validation",
            "n_obs": int(row.get("n", 0) or 0),
            "anova_p": float((anova_lookup.get(model) or {}).get("p_value")) if anova_lookup else None,
        }
        log_result(model_name=model, dataset_name="sentiment_validation", metrics=metrics, extra=extra)

C:\Users\ander\AppData\Local\Temp\ipykernel_17740\2923433470.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = [grp["ret_next"].values for _, grp in valid.groupby("signal_bucket") if len(grp) > 0]
C:\Users\ander\AppData\Local\Temp\ipykernel_17740\2923433470.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = valid.groupby("signal_bucket")["ret_next"].mean().to_dict()


,model,f_stat,p_value,mean_neg,mean_neu,mean_pos,n_obs
0,logreg_l2,1.671396,0.188323,0.000124,-0.000150,0.001782,1552
1,rf_200,0.763707,0.466111,0.001603,0.000641,-0.000025,1552


C:\TCC_USP\tcc-usp-ibovespa-sentimento\venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)


In [6]:
# 5. Correlações com defasagens (lag -2 a +2)
lag_rows = []
for model_name, sub in df.groupby("model"):
    ordered = sub.sort_values("day").set_index("day")
    for lag in range(-2, 3):
        shifted = ordered["sentiment"].shift(lag)
        aligned = pd.concat([ordered["ret_next"], shifted], axis=1).dropna()
        if len(aligned) < 4:
            corr = np.nan
            p_val = np.nan
        else:
            try:
                corr, p_val = stats.pearsonr(aligned["sentiment"], aligned["ret_next"])
            except Exception:
                corr, p_val = np.nan, np.nan
        lag_rows.append({
            "model": model_name,
            "lag": lag,
            "corr": corr,
            "p_value": p_val,
            "n": len(aligned),
        })

lag_df = pd.DataFrame(lag_rows)
lag_pivot = lag_df.pivot(index="model", columns="lag", values="corr")
display(lag_df.head())

,model,lag,corr,p_value,n
0,logreg_l2,-2,-0.031489,0.215338,1550
1,logreg_l2,-1,-0.117349,0.000004,1551
2,logreg_l2,0,0.012212,0.630699,1552
3,logreg_l2,1,-0.021934,0.388003,1551
4,logreg_l2,2,0.001031,0.967641,1550


In [7]:
# 6. Séries temporais e gráficos interativos
plots = {}

scatter_fig = px.scatter(
    df,
    x="sentiment",
    y="ret_next",
    color="model",
    trendline="ols",
    hover_data={"day": True, "signal_bucket": True},
    title="Sentimento (prob) vs Retorno D+1",
)
plots["scatter"] = scatter_fig
scatter_fig.show()

heatmap_fig = px.imshow(
    lag_pivot,
    labels=dict(x="Lag (dias)", y="Modelo", color="Correlação"),
    title="Correlação sentimento x retorno por defasagem",
    aspect="auto",
    text_auto=".2f",
)
plots["heatmap"] = heatmap_fig
heatmap_fig.show()

rolling_fig = go.Figure()
for model_name, sub in df.groupby("model"):
    ordered = sub.sort_values("day")
    ordered["sentiment_roll"] = ordered["sentiment"].rolling(5).mean()
    ordered["ret_roll"] = ordered["ret_next"].rolling(5).mean()
    rolling_fig.add_trace(
        go.Scatter(
            x=ordered["day"],
            y=ordered["sentiment_roll"],
            mode="lines",
            name=f"{model_name} - sentimento (média 5d)",
        )
    )
    rolling_fig.add_trace(
        go.Scatter(
            x=ordered["day"],
            y=ordered["ret_roll"],
            mode="lines",
            name=f"{model_name} - retorno (média 5d)",
            line=dict(dash="dash"),
            yaxis="y2",
        )
    )
rolling_fig.update_layout(
    title="Rolling 5d: sentimento x retorno",
    yaxis=dict(title="sentimento (z-score)"),
    yaxis2=dict(title="retorno", overlaying="y", side="right"),
)
plots["rolling"] = rolling_fig
rolling_fig.show()

In [8]:
# 7. Salvar tabelas e HTML dos gráficos
corr_file = os.path.join(PROC_PATH, "17_sentiment_correlations.csv")
anova_file = os.path.join(PROC_PATH, "17_sentiment_anova.csv")
lag_file = os.path.join(PROC_PATH, "17_sentiment_lags.csv")
plot_file = os.path.join(PROC_PATH, "17_sentiment_dashboard.html")

corr_df.to_csv(corr_file, index=False, encoding="utf-8")
anova_df.to_csv(anova_file, index=False, encoding="utf-8")
lag_df.to_csv(lag_file, index=False, encoding="utf-8")

with open(plot_file, "w", encoding="utf-8") as f:
    for name, fig in plots.items():
        f.write(f"<h2>{name}</h2>\n")
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn"))

print("Arquivos salvos:")
print(corr_file)
print(anova_file)
print(lag_file)
print(plot_file)


Arquivos salvos:
C:\TCC_USP\data_processed\17_sentiment_correlations.csv
C:\TCC_USP\data_processed\17_sentiment_anova.csv
C:\TCC_USP\data_processed\17_sentiment_lags.csv
C:\TCC_USP\data_processed\17_sentiment_dashboard.html


In [9]:
# 8. Resumo final
from IPython.display import Markdown

md = "**17_sentiment_validation concluido**\n"
md += f"- Modelos avaliados: {', '.join(sorted(df['model'].unique()))}\n"
md += "- Artefatos:\n"
md += "  - 17_sentiment_correlations.csv\n"
md += "  - 17_sentiment_anova.csv\n"
md += "  - 17_sentiment_lags.csv\n"
md += "  - 17_sentiment_dashboard.html\n"
md += "\n**Proximo:** notebook 18_backtest_simulation -> testar regras de trading com as probabilidades do notebook 16."

Markdown(md)


**17_sentiment_validation concluido**
- Modelos avaliados: logreg_l2, rf_200
- Artefatos:
  - 17_sentiment_correlations.csv
  - 17_sentiment_anova.csv
  - 17_sentiment_lags.csv
  - 17_sentiment_dashboard.html

**Proximo:** notebook 18_backtest_simulation -> testar regras de trading com as probabilidades do notebook 16.